In [1]:
# In[1] — thread caps (fork-safety / reproducibility)
import os
for v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
          'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[v] = '1'

In [2]:
# In[2] — imports, path, reload, confirm the passthrough is in your repo
import sys, os, importlib, warnings, inspect
warnings.simplefilter(action='ignore', category=FutureWarning)
sys.path.insert(0, os.path.abspath('../../model'))
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../.'))

In [ ]:
import os
for v in ('OMP','OPENBLAS','MKL','VECLIB','NUMEXPR'): os.environ[f'{v}_NUM_THREADS']='1'
import importlib, numpy as np, seasonal_scan_harness as ssh; importlib.reload(ssh)
FORC = ssh.build_forcings(['pre+recovery','post'], hplc_coincident=False)
RF_PRE = [0.3, 0.4, 0.5, 0.55, 0.6, 0.65]
KSZ=[0.15,0.20,0.23]; GGE=[0.25,0.28,0.31,0.33]; SIG=[0.15,0.18,0.20,0.22,0.25]; MZ=[0.05,0.10]
DETR=[(5.0,0.04),(5.0,0.067),(7.5,0.04)]                       # (w_sink,k_remin) paired -> L 125/75/188
PIN=dict(mP=0.0015, fish_fX=0.25, fish_fD=0.25, zq_fD=0.90, zq_fX=0.0, zl_fD=0.90, zl_fX=0.0)
COMBOS=[dict(KsZ=k,GGE=g,sigma_log=s,m_Z=m,w_sink=ws,k_remin=kr,**PIN)
        for k in KSZ for g in GGE for s in SIG for m in MZ for (ws,kr) in DETR]   # 360 constructs
print(f"{len(COMBOS)} constructs x {len(RF_PRE)} r_F = {len(COMBOS)*len(RF_PRE)} pre runs")
res = ssh.run_seasonal_scan(constructs=['maranon_ward'], groups=['pre+recovery'], fish_rates=RF_PRE,
    param_combos=COMBOS, years=60, spinup=15, n_harmonics=3, forcings=FORC, routed=True,
    save_path='seasonal_grid2_pre_2026-06-29.pkl', processes=None, maxtasksperchild=1)
print('pre done:', len(res.records))

In [ ]:
import os
for v in ('OMP','OPENBLAS','MKL','VECLIB','NUMEXPR'): os.environ[f'{v}_NUM_THREADS']='1'
import importlib, numpy as np, seasonal_scan_harness as ssh; importlib.reload(ssh)
FORC = ssh.build_forcings(['pre+recovery','post'], hplc_coincident=False)
RF_POST = [0.1, 0.15, 0.2]
KSZ=[0.15,0.20,0.23]; GGE=[0.25,0.28,0.31,0.33]; SIG=[0.15,0.18,0.20,0.22,0.25]; MZ=[0.05,0.10]
DETR=[(5.0,0.04),(5.0,0.067),(7.5,0.04)]
PIN=dict(mP=0.0015, fish_fX=0.25, fish_fD=0.25, zq_fD=0.90, zq_fX=0.0, zl_fD=0.90, zl_fX=0.0)
COMBOS=[dict(KsZ=k,GGE=g,sigma_log=s,m_Z=m,w_sink=ws,k_remin=kr,**PIN)
        for k in KSZ for g in GGE for s in SIG for m in MZ for (ws,kr) in DETR]
print(f"{len(COMBOS)} constructs x {len(RF_POST)} r_F = {len(COMBOS)*len(RF_POST)} post runs")
res = ssh.run_seasonal_scan(constructs=['maranon_ward'], groups=['post'], fish_rates=RF_POST,
    param_combos=COMBOS, years=60, spinup=15, n_harmonics=3, forcings=FORC, routed=True,
    save_path='seasonal_grid2_post_2026-06-29.pkl', processes=None, maxtasksperchild=1)
print('post done:', len(res.records))

In [ ]:
import numpy as np, pandas as pd, seasonal_scan_harness as ssh
from collections import defaultdict
RP=ssh.load_results('seasonal_grid2_pre_2026-06-29.pkl'); RQ=ssh.load_results('seasonal_grid2_post_2026-06-29.pkl')
OBS={'pre+recovery':RP.obs['pre+recovery'],'post':RQ.obs['post']}; obs_m=ssh.build_obs_monthly(['pre+recovery','post']); _MO=np.arange(1,13)
log_oc={g:np.log10(obs_m[g].dropna(subset=['mcs']).groupby('mo')['mcs'].median().reindex(_MO).to_numpy()) for g in OBS}
oP,oQ=OBS['pre+recovery']['med'],OBS['post']['med']
CK=['KsZ','GGE','sigma_log','m_Z','w_sink','k_remin']
SQ_THR, SHP_TOL, SHN_TOL = 0.30, 0.15, 0.25          # flag thresholds (tune freely)
ZMIN_INV, ZPRE_FLOOR, ZRAT_MAX = 0.002, 0.008, 2.0
def ck(r): return tuple(round(r[k],4) for k in CK)
def comp_e(r,o):
    ks=['mcs','pico','nano','micro']; e=[(r.get(k+'_med',np.nan)-o[k])/o[k] for k in ks if o.get(k)]
    return float(np.sqrt(np.nanmean(np.square(e))))
def brmse(r,g):
    c=np.asarray(r.get('clim_mcs',[np.nan]*12),float); return float(np.sqrt(np.nanmean((np.log10(c)-log_oc[g])**2)))
def rs(a,b): return (a-b)/a if abs(a)>1e-9 else np.nan
Qby=defaultdict(list)
for q in RQ.records: Qby[ck(q)].append(q)
rows=[]
for p in RP.records:
    if p.get('has_nan'): continue
    for q in Qby.get(ck(p),[]):
        if q.get('has_nan') or q['fish']>p['fish']+1e-9: continue
        comp=max(comp_e(p,oP),comp_e(q,oQ)); bl=max(brmse(p,'pre+recovery'),brmse(q,'post'))
        inv=q['Z200_med']-p['Z200_med']; Zrat=q['Z200_med']/max(p['Z200_med'],1e-9)
        shP=abs(rs(p['sumP_med'],q['sumP_med'])-rs(oP['sumP'],oQ['sumP'])); shN=abs(rs(p['N_med'],q['N_med'])-rs(oP['N'],oQ['N']))
        sqg=p.get('squiggle_sumP',np.nan)
        row={k:round(p[k],3) for k in CK}
        row.update(rFpre=round(p['fish'],2),rFpost=round(q['fish'],2), score=round(comp+bl,3),
            comp=round(comp,3), bloom=round(bl,3), squig=round(sqg,3), inv=round(inv,4), Zrat=round(Zrat,2),
            shP=round(shP,3), shN=round(shN,3), pze=round(p.get('pze',np.nan),2), cvPre=round(p['cv_sumP'],2),
            sq_ok=bool(sqg<=SQ_THR), Z_ok=bool(inv>=ZMIN_INV and p['Z200_med']>=ZPRE_FLOOR and Zrat<=ZRAT_MAX),
            shP_ok=bool(shP<=SHP_TOL), shN_ok=bool(shN<=SHN_TOL))
        rows.append(row)
X=pd.DataFrame(rows); X['flags']=X.sq_ok&X.Z_ok&X.shP_ok&X.shN_ok
cols=CK+['rFpre','rFpost','score','comp','bloom','squig','inv','Zrat','shP','shN','pze','cvPre','sq_ok','Z_ok','shP_ok','shN_ok']
print(f"OBS relshift sumP {rs(oP['sumP'],oQ['sumP']):.2f} N {rs(oP['N'],oQ['N']):.2f} | {len(X)} stable pairs | "
      f"flags sq<={SQ_THR} Z(inv>={ZMIN_INV},Zpre>={ZPRE_FLOOR},rat<={ZRAT_MAX}) shP<={SHP_TOL} shN<={SHN_TOL}")
print(f"\n=== TOP 25 — ALL pairs by score (comp+bloom), flags shown ===")
print(X.sort_values('score')[cols].head(25).to_string(index=False))
print(f"\n=== TOP 25 — FLAG-PASSING ({int(X.flags.sum())}/{len(X)}) by score ===")
print((X[X.flags].sort_values('score')[cols].head(25).to_string(index=False)) if X.flags.any() else "  none pass all flags — relax a threshold (likely shN) and re-read")